In [1]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
from importlib.metadata import version, PackageNotFoundError

matches = []
for path in Path("/kaggle/input").rglob("run_config.json"):
    config = json.loads(path.read_text())
    if (
        config.get("model") == "Qwen/Qwen2.5-0.5B-Instruct"
        and config.get("training_rows") == 10590
        and config.get("seed") == 42
        and (path.parent / "best_adapter").is_dir()
    ):
        matches.append(path.parent)

assert len(matches) == 1, f"Expected one baseline; found: {matches}"
BASELINE_RUN_DIR = matches[0]
BASELINE_ROOT = BASELINE_RUN_DIR.parent

for name in [
    "initial_adapter_seed42",
    "tokenized_train",
    "tokenized_validation",
    "original_data",
    "input_config.json",
    "dataset_metadata.json",
    "Qwen_ZeroShot_Validation/generation_config.json",
]:
    assert (BASELINE_ROOT / name).exists(), f"Missing: {name}"

print("Baseline:", BASELINE_ROOT)
print("Required artifacts: FOUND")

for package in [
    "torch", "transformers", "peft", "accelerate",
    "datasets", "scikit-learn", "torchao",
]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("Check complete. No model loaded or files changed.")

Baseline: /kaggle/input/datasets/muusamunufo/base-1-trained/HealthVer_Experiments
Required artifacts: FOUND
torch: 2.11.0+cu128
transformers: 5.16.1
peft: 0.20.0
accelerate: 1.14.0
datasets: 4.8.5
scikit-learn: 1.6.1
torchao: NOT INSTALLED
Check complete. No model loaded or files changed.


In [3]:
import json
import numpy as np
from pathlib import Path
from datasets import load_from_disk

TAGCOS_ROOT = Path("/kaggle/working/HealthVer_TAGCOS_Experiments")
TAGCOS_ROOT.mkdir(parents=True, exist_ok=True)

full_train = load_from_disk(
    str(BASELINE_ROOT / "tokenized_train")
)
assert len(full_train) == 10590

rng = np.random.default_rng(42)
warmup_indices = sorted(
    rng.choice(len(full_train), size=530, replace=False).tolist()
)
warmup_data = full_train.select(warmup_indices)

manifest = {
    "method": "TAGCOS_HealthVer_adaptation",
    "sampling": "uniform_without_replacement",
    "selection_seed": 42,
    "warmup_training_seed": 42,
    "warmup_epochs": 3,
    "checkpoint_policy": "each_epoch",
    "warmup_rows": 530,
    "dataset_fingerprint": full_train._fingerprint,
    "row_indices": warmup_indices,
    "ids": [str(uid) for uid in warmup_data["id"]],
}

path = TAGCOS_ROOT / "warmup_manifest.json"
if path.exists():
    assert json.loads(path.read_text()) == manifest
else:
    with path.open("x") as handle:
        json.dump(manifest, handle, indent=2)

warmup_train = warmup_data.select_columns(
    ["input_ids", "attention_mask", "labels"]
)

print("Warmup rows:", len(warmup_train))
print("Unique indices:", len(set(warmup_indices)))
print("Planned warmup epochs:", manifest["warmup_epochs"])
print("Saved:", path)
print("No model loaded. Validation and test data not accessed.")

Warmup rows: 530
Unique indices: 530
Planned warmup epochs: 3
Saved: /kaggle/working/HealthVer_TAGCOS_Experiments/warmup_manifest.json
No model loaded. Validation and test data not accessed.


In [4]:
import json
import math
import hashlib
import torch

from transformers import (
    AutoModelForCausalLM, Trainer, TrainingArguments,
    TrainerCallback, set_seed,
)
from peft import PeftModel
from torch.nn.utils.rnn import pad_sequence

DEVICE = torch.device("cuda:0")
assert torch.cuda.is_available()

WARMUP_DIR = TAGCOS_ROOT / "warmup_seed42"

input_config = json.loads(
    (BASELINE_ROOT / "input_config.json").read_text()
)
manifest = json.loads(
    (TAGCOS_ROOT / "warmup_manifest.json").read_text()
)
assert manifest["warmup_rows"] == len(warmup_train) == 530
assert manifest["warmup_epochs"] == 3

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    str(BASELINE_RUN_DIR / "best_adapter"),
    local_files_only=True,
)
tokenizer.padding_side = "right"
assert tokenizer.pad_token_id is not None

def training_collator(rows):
    padding = {
        "input_ids": tokenizer.pad_token_id,
        "attention_mask": 0,
        "labels": -100,
    }
    batch = {
        key: pad_sequence(
            [torch.tensor(row[key], dtype=torch.long) for row in rows],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding.items()
    }
    assert batch["input_ids"].shape[1] <= 384
    return batch

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class GradientCheckpointMetadata(TrainerCallback):
    def __init__(self):
        self.interval_lrs = []

    def on_step_begin(self, args, state, control, optimizer=None, **kwargs):
        # Learning rate used for this optimizer update.
        self.interval_lrs.append(float(optimizer.param_groups[0]["lr"]))

    def on_save(
        self, args, state, control,
        model=None, optimizer=None, **kwargs
    ):
        checkpoint = WARMUP_DIR / f"checkpoint-{state.global_step}"
        names_by_object = {
            id(parameter): name
            for name, parameter in model.named_parameters()
        }
        optimizer_state = optimizer.state_dict()
        parameter_names = {}

        for live_group, saved_group in zip(
            optimizer.param_groups, optimizer_state["param_groups"]
        ):
            assert len(live_group["params"]) == len(saved_group["params"])
            for parameter, saved_id in zip(
                live_group["params"], saved_group["params"]
            ):
                parameter_names[str(saved_id)] = names_by_object[id(parameter)]

        assert self.interval_lrs
        metadata = {
            "step": int(state.global_step),
            "epoch": float(state.epoch),
            "mean_interval_learning_rate": (
                sum(self.interval_lrs) / len(self.interval_lrs)
            ),
            "interval_optimizer_steps": len(self.interval_lrs),
            "optimizer_parameter_names": parameter_names,
            "model_revision": input_config["model_revision"],
            "warmup_manifest_sha256": hashlib.sha256(
                (TAGCOS_ROOT / "warmup_manifest.json").read_bytes()
            ).hexdigest(),
        }
        (checkpoint / "gradient_metadata.json").write_text(
            json.dumps(metadata, indent=2)
        )
        self.interval_lrs.clear()


def verify_completed_warmup():
    # Never overwrite or silently restart an incomplete warmup.
    required_steps = (17, 34, 51)
    state_file = WARMUP_DIR / "trainer_state.json"
    args_file = WARMUP_DIR / "training_args.json"
    metrics_file = WARMUP_DIR / "train_results.json"
    for file in (state_file, args_file, metrics_file):
        assert file.is_file(), f"Incomplete warmup; inspect before resuming: {file}"
    state = json.loads(state_file.read_text())
    saved_args = json.loads(args_file.read_text())
    saved_metrics = json.loads(metrics_file.read_text())
    assert state["global_step"] == 51 and state["epoch"] >= 3 - 1e-6
    expected_args = {
        "num_train_epochs": 3,
        "per_device_train_batch_size": 8,
        "gradient_accumulation_steps": 4,
        "learning_rate": 1e-4,
        "warmup_steps": 2,
        "lr_scheduler_type": "linear",
        "optim": "adamw_torch",
        "weight_decay": 0.01,
        "max_grad_norm": 1.0,
        "fp16": True,
        "bf16": False,
        "seed": 42,
        "data_seed": 42,
        "eval_strategy": "no",
        "save_strategy": "epoch",
    }
    for key, value in expected_args.items():
        assert saved_args[key] == value, f"Existing warmup configuration differs: {key}"
    assert saved_metrics["train_runtime"] > 0
    actual_steps = sorted(int(p.name.split("-")[-1]) for p in WARMUP_DIR.glob("checkpoint-*") if p.is_dir())
    assert actual_steps == list(required_steps)
    manifest_hash = hashlib.sha256((TAGCOS_ROOT / "warmup_manifest.json").read_bytes()).hexdigest()
    for step in required_steps:
        checkpoint = WARMUP_DIR / f"checkpoint-{step}"
        for filename in (
            "adapter_model.safetensors", "adapter_config.json", "optimizer.pt",
            "scheduler.pt", "trainer_state.json", "gradient_metadata.json",
        ):
            file = checkpoint / filename
            assert file.is_file() and file.stat().st_size > 0, file
        info = json.loads((checkpoint / "gradient_metadata.json").read_text())
        checkpoint_state = json.loads((checkpoint / "trainer_state.json").read_text())
        assert checkpoint_state["global_step"] == info["step"] == step
        assert info["epoch"] == step / 17
        assert info["interval_optimizer_steps"] == 17
        assert info["mean_interval_learning_rate"] > 0
        assert info["optimizer_parameter_names"]
        assert info["model_revision"] == input_config["model_revision"]
        assert info["warmup_manifest_sha256"] == manifest_hash
    print("Verified completed warmup; reusing checkpoints 17, 34 and 51.")

if WARMUP_DIR.exists():
    verify_completed_warmup()
else:
    set_seed(42)
    warmup_base = AutoModelForCausalLM.from_pretrained(
        input_config["model"],
        revision=input_config["model_revision"],
        dtype=torch.float16,
        attn_implementation="sdpa",
    ).to(DEVICE)

    warmup_model = PeftModel.from_pretrained(
        warmup_base,
        str(BASELINE_ROOT / "initial_adapter_seed42"),
        is_trainable=True,
    )
    del warmup_base

    warmup_model.config.use_cache = False
    warmup_model.config.pad_token_id = tokenizer.pad_token_id
    warmup_model.gradient_checkpointing_disable()

    for name, parameter in warmup_model.named_parameters():
        if parameter.requires_grad:
            assert "lora_" in name
            parameter.data = parameter.data.float()
    del parameter

    assert sum(
        p.numel() for p in warmup_model.parameters() if p.requires_grad
    ) == 8798208

    steps_per_epoch = math.ceil(len(warmup_train) / 32)

    warmup_args = SingleGPUArguments(
        output_dir=str(WARMUP_DIR),
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        warmup_steps=math.ceil(0.03 * 3 * steps_per_epoch),
        lr_scheduler_type="linear",
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=1.0,
        fp16=True,
        bf16=False,
        gradient_checkpointing=False,
        eval_strategy="no",
        save_strategy="epoch",
        save_total_limit=3,
        save_only_model=False,
        logging_steps=10,
        logging_first_step=True,
        report_to="none",
        dataloader_num_workers=0,
        seed=42,
        data_seed=42,
        label_names=["labels"],
    )

    assert warmup_args.world_size == 1
    assert warmup_args.device == DEVICE
    assert warmup_args.train_batch_size == 8

    warmup_trainer = Trainer(
        model=warmup_model,
        args=warmup_args,
        train_dataset=warmup_train,
        data_collator=training_collator,
        processing_class=tokenizer,
        callbacks=[GradientCheckpointMetadata()],
    )

    WARMUP_DIR.mkdir(parents=True, exist_ok=True)
    (WARMUP_DIR / "training_args.json").write_text(
        warmup_args.to_json_string()
    )

    print("Expected optimizer steps:", 3 * steps_per_epoch)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    result = warmup_trainer.train()

    metrics = dict(result.metrics)
    metrics["peak_allocated_gib"] = (
        torch.cuda.max_memory_allocated(DEVICE) / 1024**3
    )
    warmup_trainer.save_metrics("train", metrics)
    warmup_trainer.save_state()

    checkpoints = sorted(
        WARMUP_DIR.glob("checkpoint-*"),
        key=lambda path: int(path.name.split("-")[-1]),
    )
    assert len(checkpoints) == 3

    for checkpoint in checkpoints:
        for filename in [
            "adapter_model.safetensors", "adapter_config.json",
            "optimizer.pt", "scheduler.pt",
            "trainer_state.json", "gradient_metadata.json",
        ]:
            assert (checkpoint / filename).is_file(), (
                f"Missing: {checkpoint / filename}"
            )
        print("Verified checkpoint:", checkpoint.name)

    print(json.dumps(metrics, indent=2))
    print("Warmup complete. No validation or test evaluation performed.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Expected optimizer steps: 51


Step,Training Loss
1,0.335295
10,0.514210
20,0.335162
30,0.275952
40,0.254792
50,0.228620


Verified checkpoint: checkpoint-17
Verified checkpoint: checkpoint-34
Verified checkpoint: checkpoint-51
{
  "train_runtime": 135.4039,
  "train_samples_per_second": 11.743,
  "train_steps_per_second": 0.377,
  "total_flos": 721662758115840.0,
  "train_loss": 0.31590137236258564,
  "epoch": 3.0,
  "peak_allocated_gib": 9.543006896972656
}
Warmup complete. No validation or test evaluation performed.


In [5]:
import gc
import json
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

FEATURE_CHECKPOINT = WARMUP_DIR / "checkpoint-17"
metadata = json.loads(
    (FEATURE_CHECKPOINT / "gradient_metadata.json").read_text()
)
assert metadata["step"] == 17
assert metadata["model_revision"] == input_config["model_revision"]

# Release the completed warmup model and optimizer.
for object_name in ["warmup_trainer", "warmup_model"]:
    globals().pop(object_name, None)
gc.collect()
torch.cuda.empty_cache()

feature_base = AutoModelForCausalLM.from_pretrained(
    input_config["model"],
    revision=input_config["model_revision"],
    dtype=torch.float16,
    attn_implementation="sdpa",
).to(DEVICE)

feature_model = PeftModel.from_pretrained(
    feature_base,
    str(FEATURE_CHECKPOINT),
    is_trainable=True,
)
del feature_base

feature_model.config.use_cache = False
feature_model.config.pad_token_id = tokenizer.pad_token_id
feature_model.gradient_checkpointing_disable()
feature_model.eval()  # Disable dropout during feature extraction.

feature_parameters = [
    (name, parameter)
    for name, parameter in feature_model.named_parameters()
    if parameter.requires_grad
]
for name, parameter in feature_parameters:
    assert "lora_" in name
    parameter.data = parameter.data.float()
del parameter

optimizer_snapshot = torch.load(
    FEATURE_CHECKPOINT / "optimizer.pt",
    map_location="cpu",
    weights_only=True,
)
name_to_id = {
    name: int(saved_id)
    for saved_id, name in metadata["optimizer_parameter_names"].items()
}
assert set(name_to_id) == {name for name, _ in feature_parameters}

moment_parts, variance_parts = [], []

for group in optimizer_snapshot["param_groups"]:
    assert tuple(group["betas"]) == (0.9, 0.999)
    assert group["eps"] == 1e-8

for name, parameter in feature_parameters:
    state = optimizer_snapshot["state"][name_to_id[name]]
    assert int(state["step"]) == 17
    assert state["exp_avg"].shape == parameter.shape
    assert state["exp_avg_sq"].shape == parameter.shape
    moment_parts.append(state["exp_avg"].float().flatten())
    variance_parts.append(state["exp_avg_sq"].float().flatten())

adam_m = torch.cat(moment_parts).to(DEVICE)
adam_v = torch.cat(variance_parts).to(DEVICE)
assert adam_m.numel() == adam_v.numel() == 8798208
assert torch.isfinite(adam_m).all()
assert torch.isfinite(adam_v).all() and (adam_v >= 0).all()

del optimizer_snapshot, moment_parts, variance_parts, state, parameter
gc.collect()

def adam_feature(row):
    feature_model.zero_grad(set_to_none=True)
    batch = {
        key: value.to(DEVICE)
        for key, value in training_collator([row]).items()
    }

    # Fixed loss scaling reduces FP16 backward underflow.
    # Undo it before applying the reference feature formula.
    loss_scale = 1024.0
    with torch.enable_grad():
        with torch.autocast("cuda", dtype=torch.float16):
            loss = feature_model(**batch).loss
        assert torch.isfinite(loss)
        (loss * loss_scale).backward()

    assert all(p.grad is not None for _, p in feature_parameters)
    gradient = torch.cat([
        p.grad.detach().float().flatten()
        for _, p in feature_parameters
    ]) / loss_scale
    assert torch.isfinite(gradient).all()

    feature = (
        0.9 * adam_m + 0.1 * gradient
    ) / torch.sqrt(
        0.999 * adam_v + 0.001 * gradient.square() + 1e-8
    )
    assert torch.isfinite(feature).all()

    diagnostics = {
        "loss": float(loss.detach()),
        "gradient_norm": float(gradient.norm()),
        "adam_feature_norm": float(feature.norm()),
        "feature_dimensions": feature.numel(),
    }
    feature_model.zero_grad(set_to_none=True)
    return feature.detach(), diagnostics

for row_index in range(3):
    feature, diagnostics = adam_feature(full_train[row_index])
    assert diagnostics["gradient_norm"] > 0
    assert diagnostics["adam_feature_norm"] > 0
    print({"row_index": row_index, **diagnostics})
    del feature

print("Checkpoint-17 Adam-feature sanity check passed.")
print("No optimizer updates, projection or full extraction performed.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'row_index': 0, 'loss': 0.2703084349632263, 'gradient_norm': 8.553690910339355, 'adam_feature_norm': 2901.412841796875, 'feature_dimensions': 8798208}
{'row_index': 1, 'loss': 0.17633391916751862, 'gradient_norm': 6.780211925506592, 'adam_feature_norm': 2800.62744140625, 'feature_dimensions': 8798208}
{'row_index': 2, 'loss': 0.45468008518218994, 'gradient_norm': 7.438715934753418, 'adam_feature_norm': 2419.63916015625, 'feature_dimensions': 8798208}
Checkpoint-17 Adam-feature sanity check passed.
No optimizer updates, projection or full extraction performed.


In [6]:
import sys
import shutil
import importlib.metadata as package_metadata
import torch

print("Python:", sys.version.split()[0])
print("CUDA toolkit compiler:", shutil.which("nvcc"))

for package in ("traker", "fast-jl"):
    try:
        print(f"{package}: {package_metadata.version(package)}")
    except package_metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

device = torch.device("cuda:0")
properties = torch.cuda.get_device_properties(device)
print("GPU:", properties.name)
print("Compute capability:", torch.cuda.get_device_capability(device))

try:
    import fast_jl

    # Small backend smoke test; independent of training randomness.
    generator = torch.Generator(device=device).manual_seed(0)
    probe = torch.randn(
        8, 1000, device=device,
        dtype=torch.float32, generator=generator,
    )

    projected = fast_jl.project_rademacher_8(
        probe, 512, 0, properties.multi_processor_count
    )
    torch.cuda.synchronize(device)

    assert projected.shape == (8, 512)
    assert torch.isfinite(projected).all()
    assert (projected.float().norm(dim=1) > 0).all()

    print("Accelerated projection backend: PASSED")
    del probe, projected

except Exception as error:
    print("Accelerated projection backend: UNAVAILABLE")
    print(type(error).__name__, str(error)[:800])

print("Check complete. No installation, training, or full extraction.")

Python: 3.13.15
CUDA toolkit compiler: /usr/local/cuda/bin/nvcc
traker: NOT INSTALLED
fast-jl: NOT INSTALLED
GPU: Tesla T4
Compute capability: (7, 5)
Accelerated projection backend: UNAVAILABLE
ModuleNotFoundError No module named 'fast_jl'
Check complete. No installation, training, or full extraction.


In [7]:
import os
import sys
import subprocess
import importlib
import importlib.metadata as metadata

protected = (
    "torch", "transformers", "peft",
    "accelerate", "datasets", "scikit-learn",
)
before = {name: metadata.version(name) for name in protected}

build_env = os.environ.copy()
build_env["MAX_JOBS"] = "2"
build_env["TORCH_CUDA_ARCH_LIST"] = "7.5"

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--no-build-isolation",
        "traker==0.3.2",
        "fast-jl==0.1.3",
    ],
    env=build_env,
)

importlib.invalidate_caches()

after = {name: metadata.version(name) for name in protected}
assert before == after, "Training package versions changed."

import torch
import fast_jl
from trak.projectors import CudaProjector

device = torch.device("cuda:0")
generator = torch.Generator(device=device).manual_seed(0)
probe = torch.randn(
    8, 1000, device=device,
    dtype=torch.float32, generator=generator,
)
projected = fast_jl.project_rademacher_8(
    probe, 512, 0,
    torch.cuda.get_device_properties(device).multi_processor_count,
)
torch.cuda.synchronize(device)

assert projected.shape == (8, 512)
assert torch.isfinite(projected).all()
assert (projected.float().norm(dim=1) > 0).all()

print("traker:", metadata.version("traker"))
print("fast-jl:", metadata.version("fast-jl"))
print("Training package versions unchanged.")
print("CUDA projection smoke test: PASSED")
del probe, projected

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for traker: filename=traker-0.3.2-py3-none-any.whl size=29027 sha256=109f7925971171227887cc49f73e1dae4997d360710f190a832aeda9ecf68075
  Stored in directory: /root/.cache/pip/wheels/94/5d/b7/e2e6ee7a0793141139d942c5da6ef8dd532277a97fe31e8c7c
  Created wheel for fast-jl: filename=fast_jl-0.1.3-cp313-cp313-linux_x86_64.whl size=948387 sha256=718d2c24518843b6789ad3edb57ad950cd5c8399e351a58d6170f19739601f98
  Stored in directory: /root/.cache/pip/wheels/8e/be/a0/9f21db120dc77b1ef68768d6ef29db83f927792efe5c8a1a1e
Successfully built traker fast-jl
traker: 0.3.2
fast-jl: 0.1.3
Training package versions unchanged.
CUDA projection smoke test: PASSED


In [8]:
import time
import json
import numpy as np
import torch
from trak.projectors import CudaProjector, ProjectionType

PROJECTION_DIM = 8192
PROJECTION_SEED = 0
PROJECTION_BATCH_SIZE = 16

feature_model.eval()
dimension = sum(p.numel() for _, p in feature_parameters)
assert dimension == 8798208

tagcos_projector = CudaProjector(
    grad_dim=dimension,
    proj_dim=PROJECTION_DIM,
    seed=PROJECTION_SEED,
    proj_type=ProjectionType.rademacher,
    device=DEVICE,
    max_batch_size=PROJECTION_BATCH_SIZE,
)

indices = np.random.default_rng(123).choice(
    len(full_train), PROJECTION_BATCH_SIZE, replace=False
).tolist()

torch.cuda.synchronize(DEVICE)
torch.cuda.reset_peak_memory_stats(DEVICE)
started = time.perf_counter()

features = torch.empty(
    PROJECTION_BATCH_SIZE, dimension,
    device=DEVICE, dtype=torch.float16,
)
for position, index in enumerate(indices):
    vector, diagnostics = adam_feature(full_train[index])
    features[position].copy_(vector)
    del vector

assert torch.isfinite(features).all()
torch.cuda.synchronize(DEVICE)
feature_seconds = time.perf_counter() - started

started = time.perf_counter()
with torch.no_grad():
    projected = tagcos_projector.project(features, model_id=0).float()
torch.cuda.synchronize(DEVICE)
projection_seconds = time.perf_counter() - started

assert projected.shape == (PROJECTION_BATCH_SIZE, PROJECTION_DIM)
assert torch.isfinite(projected).all()
norms = projected.norm(dim=1, keepdim=True)
assert (norms > 0).all()

with torch.no_grad():
    repeated = tagcos_projector.project(features, model_id=0).float()
torch.testing.assert_close(projected, repeated, rtol=1e-5, atol=1e-5)

normalized = projected / norms
torch.testing.assert_close(
    normalized.norm(dim=1),
    torch.ones(PROJECTION_BATCH_SIZE, device=DEVICE),
    rtol=1e-5, atol=1e-5,
)

print(json.dumps({
    "examples": PROJECTION_BATCH_SIZE,
    "projected_dimensions": PROJECTION_DIM,
    "feature_seconds": feature_seconds,
    "projection_seconds": projection_seconds,
    "rough_three_checkpoint_minutes": (
        (feature_seconds + projection_seconds)
        * len(full_train) / PROJECTION_BATCH_SIZE * 3 / 60
    ),
    "peak_allocated_gib": (
        torch.cuda.max_memory_allocated(DEVICE) / 1024**3
    ),
    "repeatability": "PASSED",
}, indent=2))

del features, projected, repeated, normalized, norms
print("Benchmark complete. No optimizer updates or files written.")

{
  "examples": 16,
  "projected_dimensions": 8192,
  "feature_seconds": 3.19433503099998,
  "projection_seconds": 0.2494092230000433,
  "rough_three_checkpoint_minutes": 113.96641140581328,
  "peak_allocated_gib": 2.0331950187683105,
  "repeatability": "PASSED"
}
Benchmark complete. No optimizer updates or files written.


In [9]:
import gc, os, json, math, time, hashlib
from pathlib import Path
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from torch.nn.utils.rnn import pad_sequence
from trak.projectors import CudaProjector, ProjectionType
from tqdm.auto import tqdm

BASELINE_ROOT = Path('/kaggle/input/datasets/muusamunufo/base-1-trained/HealthVer_Experiments')
BASELINE_RUN_DIR = BASELINE_ROOT / 'Qwen_LoRA_full_seed42'
TAGCOS_ROOT = Path('/kaggle/working/HealthVer_TAGCOS_Experiments')
WARMUP_DIR = TAGCOS_ROOT / 'warmup_seed42'
DEVICE = torch.device('cuda:0')
assert torch.cuda.is_available()
torch.cuda.set_device(DEVICE)
CHECKPOINT_STEPS = (17, 34, 51)
SUBSET_SIZES = {'50pct':5295, '25pct':2648, '10pct':1059, '5pct':530}

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False))
    os.replace(tmp, path)

def json_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()

input_config = json.loads((BASELINE_ROOT / 'input_config.json').read_text())
baseline_config = json.loads((BASELINE_RUN_DIR / 'run_config.json').read_text())
full_train = load_from_disk(str(BASELINE_ROOT / 'tokenized_train'))
warmup_manifest = json.loads((TAGCOS_ROOT / 'warmup_manifest.json').read_text())
assert len(full_train) == 10590
assert full_train._fingerprint == '28b24c0106f60948'
assert input_config['model_revision'] == '7ae557604adf67be50417f59c2c2f167def9a775'
assert warmup_manifest['dataset_fingerprint'] == full_train._fingerprint
assert len(warmup_manifest['row_indices']) == len(set(warmup_manifest['row_indices'])) == 530
assert warmup_manifest['ids'] == [str(x) for x in full_train.select(warmup_manifest['row_indices'])['id']]
assert warmup_manifest['warmup_epochs'] == 3
assert warmup_manifest['warmup_training_seed'] == 42
assert warmup_manifest['selection_seed'] == 42

versions = {p: package_metadata.version(p) for p in (
    'torch','transformers','peft','accelerate','datasets','scikit-learn','scipy','traker','fast-jl')}
assert versions['traker'] == '0.3.2' and versions['fast-jl'] == '0.1.3'
save_json(TAGCOS_ROOT / 'continuation_environment.json', versions)
for step in CHECKPOINT_STEPS:
    folder = WARMUP_DIR / f'checkpoint-{step}'
    for name in ('adapter_config.json','adapter_model.safetensors',
                 'optimizer.pt','gradient_metadata.json'):
        assert (folder / name).is_file(), folder / name
    info = json.loads((folder / 'gradient_metadata.json').read_text())
    assert info['step'] == step
    assert info['model_revision'] == input_config['model_revision']
    assert info['warmup_manifest_sha256'] == sha256(TAGCOS_ROOT / 'warmup_manifest.json')
    assert info['mean_interval_learning_rate'] > 0

TAGCOS_CONFIG = {
    'method':'TAGCOS_HealthVer_adaptation',
    'reference_commit':'93a5a38681323e21aa9b6d503674bef814d7916c',
    'warmup_manifest_sha256':sha256(TAGCOS_ROOT / 'warmup_manifest.json'),
    'checkpoints':list(CHECKPOINT_STEPS),
    'adam_feature':'(0.9*m+0.1*g)/sqrt(0.999*v+0.001*g*g+1e-8)',
    'moment_bias_correction':False,
    'feature_model_mode':'eval', 'backward_loss_scale':1024.0,
    'projection':'CudaProjector_rademacher', 'projection_dim':8192,
    'projection_seed':0, 'projection_model_id':0, 'projection_batch_size':16,
    'projection_input_dtype':'float16','feature_storage_dtype':'float32',
    'checkpoint_normalization':'row_L2',
    'aggregation':'normalized_mean_interval_learning_rate_weights',
    'renormalize_after_aggregation':False,
    'kmeans':{'n_clusters':100,'random_state':42,'n_init':10,
              'algorithm':'lloyd','max_iter':300,'tol':1e-4},
    'cluster_budget':'largest_remainder_ties_by_cluster_id',
    'omp':{'positive':True,'lambda':1.0,'tol':1e-4,
           'implementation':'numpy_float32_scipy_gelsy',
           'first_atom_unregularized':True,
           'negative_coefficient_handling':'remove_and_refit',
           'quota_completion':'repeat_on_remaining_against_original_centroid',
           'no_progress':'raise_no_random_fallback'},
    'subset_sizes':SUBSET_SIZES,'nested_subsets_required':False,
    'final_training_seed':42,'final_training_epochs':3,
    'final_training_weights':'uniform',
    'checkpoint_selection':'validation_macro_f1',
    'final_training_init':'initial_adapter_seed42',
    'scikit_learn':versions['scikit-learn'],'scipy':versions['scipy'],'traker':versions['traker'],
    'fast_jl':versions['fast-jl'],
}
protocol_path = TAGCOS_ROOT / 'tagcos_protocol.json'
if protocol_path.exists():
    assert json.loads(protocol_path.read_text()) == TAGCOS_CONFIG, 'Protocol changed.'
else:
    save_json(protocol_path, TAGCOS_CONFIG)
PROTOCOL_HASH = json_hash(TAGCOS_CONFIG)
print('Protocol saved. Training rows:', len(full_train))
print('Checkpoints:', CHECKPOINT_STEPS, '| budgets:', SUBSET_SIZES)


Protocol saved. Training rows: 10590
Checkpoints: (17, 34, 51) | budgets: {'50pct': 5295, '25pct': 2648, '10pct': 1059, '5pct': 530}


In [10]:
# Release references held by the earlier interactive cells.
for object_name in ('feature_model','feature_parameters','adam_m','adam_v',
                    'feature_base','base_model','base','model','trainer',
                    'warmup_model','warmup_trainer','parameter','vector',
                    'features','buffer','projected','repeated','normalized'):
    globals().pop(object_name, None)
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(str(BASELINE_RUN_DIR / 'best_adapter'))
tokenizer.padding_side = 'right'
assert tokenizer.pad_token_id is not None

def feature_collator(row):
    return {k:torch.tensor([row[k]], dtype=torch.long, device=DEVICE)
            for k in ('input_ids','attention_mask','labels')}

def inspect_features(path, unit_norm=True):
    arr = np.load(path, mmap_mode='r', allow_pickle=False)
    assert arr.shape == (len(full_train), 8192) and arr.dtype == np.float32
    for start in range(0, len(arr), 512):
        batch = np.asarray(arr[start:start+512])
        assert np.isfinite(batch).all(), (path, start)
        if unit_norm:
            assert np.allclose(np.linalg.norm(batch,axis=1),1.0,rtol=1e-4,atol=1e-5)
    del arr

def checkpoint_feature_identity(step):
    ckpt = WARMUP_DIR / f'checkpoint-{step}'
    return {
        'protocol_sha256':PROTOCOL_HASH,
        'dataset_fingerprint':full_train._fingerprint,
        'checkpoint_step':step,
        'model_revision':input_config['model_revision'],
        'adapter_sha256':sha256(ckpt / 'adapter_model.safetensors'),
        'optimizer_sha256':sha256(ckpt / 'optimizer.pt'),
        'gradient_metadata_sha256':sha256(ckpt / 'gradient_metadata.json'),
    }

def extract_checkpoint(step):
    checkpoint = WARMUP_DIR / f'checkpoint-{step}'
    directory = TAGCOS_ROOT / 'features' / checkpoint.name
    directory.mkdir(parents=True, exist_ok=True)
    final = directory / 'normalized_features.npy'
    meta_path = directory / 'feature_metadata.json'
    identity_path = directory / 'feature_identity.json'
    partial = directory / 'normalized_features.resume.npy'
    progress_path = directory / 'extraction_progress.json'
    identity = checkpoint_feature_identity(step)
    checkpoint_meta = json.loads((checkpoint / 'gradient_metadata.json').read_text())

    if final.exists():
        assert meta_path.exists(), f'Array without completion metadata: {directory}'
        saved = json.loads(meta_path.read_text())
        expected = {'rows':len(full_train),'dimensions':8192,'projection_seed':0,
                    'projection_model_id':0,'projection_batch_size':16,
                    'projection':'CudaProjector_rademacher',
                    'projection_input_dtype':'float16','storage_dtype':'float32',
                    'normalization':'per_checkpoint_row_L2',
                    'row_order':'original_tokenized_train_order',
                    'dataset_fingerprint':full_train._fingerprint}
        for k,v in expected.items():
            assert saved[k] == v, (directory,k)
        assert Path(saved['checkpoint']).name == checkpoint.name
        assert saved['mean_interval_learning_rate'] == checkpoint_meta['mean_interval_learning_rate']
        inspect_features(final)
        if identity_path.exists():
            receipt = json.loads(identity_path.read_text())
            assert receipt['identity'] == identity
            assert receipt['features_sha256'] == sha256(final)
        else:
            # Only the earlier checkpoint-17 cell is allowed as a legacy output.
            assert step == 17 and 'protocol_sha256' not in saved
            save_json(identity_path, {'identity':identity,'features_sha256':sha256(final),
                'provenance':'adopted_prior_checkpoint17_output_after_metadata_and_array_checks'})
        print(f'{checkpoint.name}: verified completed features; skipped.')
        return

    progress = None
    if progress_path.exists():
        progress = json.loads(progress_path.read_text())
        assert progress['identity'] == identity, 'Partial extraction configuration changed.'
        assert partial.exists() and not meta_path.exists()
    else:
        assert not meta_path.exists() and not identity_path.exists(), 'Incomplete completion metadata.'
    done = int(progress['completed_rows']) if progress else 0
    previous_seconds = float(progress['extraction_seconds']) if progress else 0.0
    assert 0 <= done <= len(full_train)
    assert done % 16 == 0 or done == len(full_train)
    mmap = np.lib.format.open_memmap(partial, mode='r+' if progress else 'w+',
        dtype=np.float32, shape=(len(full_train),8192))
    assert mmap.shape == (len(full_train),8192) and mmap.dtype == np.float32
    if done:
        assert np.isfinite(mmap[:done]).all()
        assert np.allclose(np.linalg.norm(mmap[:done],axis=1),1,rtol=1e-4,atol=1e-5)

    base = model = moments = projector = buf = None
    m = v = g = u = batch = loss = projected = norms = None
    params = []
    started = None
    try:
        base = AutoModelForCausalLM.from_pretrained(
            input_config['model'],revision=input_config['model_revision'],
            dtype=torch.float16,attn_implementation='sdpa').to(DEVICE)
        model = PeftModel.from_pretrained(base, str(checkpoint), is_trainable=True)
        model.config.use_cache = False
        model.config.pad_token_id = tokenizer.pad_token_id
        model.gradient_checkpointing_disable()
        model.eval()
        params = [(name,p) for name,p in model.named_parameters() if p.requires_grad]
        for name,p in params:
            assert 'lora_' in name
            p.data = p.data.float()
        del p
        assert sum(p.numel() for _,p in params) == 8798208
        name_to_id = {name:int(sid) for sid,name in checkpoint_meta['optimizer_parameter_names'].items()}
        assert set(name_to_id) == {name for name,_ in params}
        moments = torch.load(checkpoint/'optimizer.pt',map_location='cpu',weights_only=True)
        for group in moments['param_groups']:
            assert tuple(group['betas']) == (0.9,0.999) and group['eps'] == 1e-8
        first,second = [],[]
        for name,p in params:
            state = moments['state'][name_to_id[name]]
            assert int(state['step']) == step
            assert state['exp_avg'].shape == state['exp_avg_sq'].shape == p.shape
            first.append(state['exp_avg'].float().flatten())
            second.append(state['exp_avg_sq'].float().flatten())
        m = torch.cat(first).to(DEVICE)
        v = torch.cat(second).to(DEVICE)
        assert torch.isfinite(m).all() and torch.isfinite(v).all() and (v >= 0).all()
        del first,second,state,p
        moments = None
        parameter_order_sha256 = json_hash([(name,list(p.shape)) for name,p in params])
        projector = CudaProjector(grad_dim=m.numel(),proj_dim=8192,seed=0,
            proj_type=ProjectionType.rademacher,device=DEVICE,max_batch_size=16)
        buf = torch.empty((16,m.numel()),dtype=torch.float16,device=DEVICE)
        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        started = time.perf_counter()

        for start in tqdm(range(done,len(full_train),16),desc=f'{checkpoint.name} features'):
            stop = min(start+16,len(full_train))
            buf.zero_()
            for pos,index in enumerate(range(start,stop)):
                model.zero_grad(set_to_none=True)
                batch = feature_collator(full_train[index])
                assert batch['input_ids'].shape[1] <= 384
                assert (batch['labels'] != -100).any()
                with torch.enable_grad(), torch.autocast('cuda',dtype=torch.float16):
                    loss = model(**batch).loss
                assert torch.isfinite(loss), index
                (loss * 1024.0).backward()
                assert all(p.grad is not None for _,p in params)
                g = torch.cat([p.grad.detach().float().flatten() for _,p in params]) / 1024.0
                assert torch.isfinite(g).all(), index
                u = (0.9*m + 0.1*g) / torch.sqrt(0.999*v + 0.001*g.square() + 1e-8)
                assert torch.isfinite(u).all(), index
                buf[pos].copy_(u)
                del g,u,loss,batch
            model.zero_grad(set_to_none=True)
            assert torch.isfinite(buf).all()
            with torch.no_grad():
                projected = projector.project(buf,model_id=0)[:stop-start].float()
                assert torch.isfinite(projected).all()
                norms = projected.norm(dim=1,keepdim=True)
                assert torch.isfinite(norms).all() and (norms > 0).all()
                mmap[start:stop] = (projected/norms).cpu().numpy()
            del projected,norms
            done = stop
            if done % 256 == 0 or done == len(full_train):
                mmap.flush()
                save_json(progress_path,{'identity':identity,'completed_rows':done,
                    'extraction_seconds':previous_seconds+time.perf_counter()-started,
                    'parameter_order_sha256':parameter_order_sha256})
        torch.cuda.synchronize(DEVICE)
        elapsed = previous_seconds + time.perf_counter() - started
        mmap.flush()
        del mmap
        inspect_features(partial)
        report = {
            'checkpoint':str(checkpoint),'rows':len(full_train),'dimensions':8192,
            'projection_seed':0,'projection_model_id':0,'projection_batch_size':16,
            'projection':'CudaProjector_rademacher','projection_input_dtype':'float16',
            'storage_dtype':'float32','normalization':'per_checkpoint_row_L2',
            'row_order':'original_tokenized_train_order',
            'dataset_fingerprint':full_train._fingerprint,
            'mean_interval_learning_rate':checkpoint_meta['mean_interval_learning_rate'],
            'extraction_seconds':elapsed,
            'peak_allocated_gib':torch.cuda.max_memory_allocated(DEVICE)/1024**3,
            'protocol_sha256':PROTOCOL_HASH,'parameter_order_sha256':parameter_order_sha256,
        }
        os.replace(partial,final)
        save_json(meta_path,report)
        save_json(identity_path,{'identity':identity,'features_sha256':sha256(final),
            'provenance':'continuation_extraction'})
        print(json.dumps(report,indent=2))
    finally:
        params.clear()
        base = model = moments = projector = buf = None
        # Clear local GPU tensors even when an assertion interrupts extraction.
        m = v = g = u = batch = loss = projected = norms = None
        gc.collect()
        torch.cuda.empty_cache()

for checkpoint_step in CHECKPOINT_STEPS:
    extract_checkpoint(checkpoint_step)
    gc.collect()
    torch.cuda.empty_cache()
print('All three checkpoint feature files complete.')


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

checkpoint-17 features:   0%|          | 0/662 [00:00<?, ?it/s]

{
  "checkpoint": "/kaggle/working/HealthVer_TAGCOS_Experiments/warmup_seed42/checkpoint-17",
  "rows": 10590,
  "dimensions": 8192,
  "projection_seed": 0,
  "projection_model_id": 0,
  "projection_batch_size": 16,
  "projection": "CudaProjector_rademacher",
  "projection_input_dtype": "float16",
  "storage_dtype": "float32",
  "normalization": "per_checkpoint_row_L2",
  "row_order": "original_tokenized_train_order",
  "dataset_fingerprint": "28b24c0106f60948",
  "mean_interval_learning_rate": 7.857142857142858e-05,
  "extraction_seconds": 2246.903090412,
  "peak_allocated_gib": 2.541429042816162,
  "protocol_sha256": "50ed943149137753976ffd395d6b91a07c162fe4f03499faa6e36744a3de1180",
  "parameter_order_sha256": "bb6228bc4a69b852db22e4a96880a657a4a13b6176c769a72c10ab7a906d9c1e"
}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

checkpoint-34 features:   0%|          | 0/662 [00:00<?, ?it/s]

{
  "checkpoint": "/kaggle/working/HealthVer_TAGCOS_Experiments/warmup_seed42/checkpoint-34",
  "rows": 10590,
  "dimensions": 8192,
  "projection_seed": 0,
  "projection_model_id": 0,
  "projection_batch_size": 16,
  "projection": "CudaProjector_rademacher",
  "projection_input_dtype": "float16",
  "storage_dtype": "float32",
  "normalization": "per_checkpoint_row_L2",
  "row_order": "original_tokenized_train_order",
  "dataset_fingerprint": "28b24c0106f60948",
  "mean_interval_learning_rate": 5.306122448979592e-05,
  "extraction_seconds": 2253.8674637760005,
  "peak_allocated_gib": 2.541429042816162,
  "protocol_sha256": "50ed943149137753976ffd395d6b91a07c162fe4f03499faa6e36744a3de1180",
  "parameter_order_sha256": "bb6228bc4a69b852db22e4a96880a657a4a13b6176c769a72c10ab7a906d9c1e"
}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

checkpoint-51 features:   0%|          | 0/662 [00:00<?, ?it/s]

{
  "checkpoint": "/kaggle/working/HealthVer_TAGCOS_Experiments/warmup_seed42/checkpoint-51",
  "rows": 10590,
  "dimensions": 8192,
  "projection_seed": 0,
  "projection_model_id": 0,
  "projection_batch_size": 16,
  "projection": "CudaProjector_rademacher",
  "projection_input_dtype": "float16",
  "storage_dtype": "float32",
  "normalization": "per_checkpoint_row_L2",
  "row_order": "original_tokenized_train_order",
  "dataset_fingerprint": "28b24c0106f60948",
  "mean_interval_learning_rate": 1.836734693877551e-05,
  "extraction_seconds": 2272.3630383259997,
  "peak_allocated_gib": 2.541429042816162,
  "protocol_sha256": "50ed943149137753976ffd395d6b91a07c162fe4f03499faa6e36744a3de1180",
  "parameter_order_sha256": "bb6228bc4a69b852db22e4a96880a657a4a13b6176c769a72c10ab7a906d9c1e"
}
All three checkpoint feature files complete.


In [11]:
feature_paths = [TAGCOS_ROOT/'features'/f'checkpoint-{s}'/'normalized_features.npy'
                 for s in CHECKPOINT_STEPS]
feature_reports = [json.loads(p.with_name('feature_metadata.json').read_text()) for p in feature_paths]
learning_rates = np.array([r['mean_interval_learning_rate'] for r in feature_reports],dtype=np.float64)
assert np.isfinite(learning_rates).all() and (learning_rates > 0).all()
checkpoint_weights = learning_rates / learning_rates.sum()
aggregate_path = TAGCOS_ROOT / 'aggregated_features.npy'
aggregate_meta_path = TAGCOS_ROOT / 'aggregation_metadata.json'
aggregation_identity = {
    'protocol_sha256':PROTOCOL_HASH,'dataset_fingerprint':full_train._fingerprint,
    'input_sha256':[sha256(p) for p in feature_paths],
    'checkpoint_steps':list(CHECKPOINT_STEPS),
    'checkpoint_weights':checkpoint_weights.tolist(),
}
if aggregate_meta_path.exists():
    aggregation_metadata = json.loads(aggregate_meta_path.read_text())
    assert aggregation_metadata['identity'] == aggregation_identity
    assert aggregation_metadata['sha256'] == sha256(aggregate_path)
    inspect_features(aggregate_path,unit_norm=False)
    print('Verified existing aggregation.')
else:
    assert not aggregate_path.exists(), 'Aggregation array lacks completion metadata.'
    started = time.perf_counter()
    arrays = [np.load(p,mmap_mode='r',allow_pickle=False) for p in feature_paths]
    tmp = aggregate_path.with_name('aggregated_features.partial.npy')
    out = np.lib.format.open_memmap(tmp,mode='w+',dtype=np.float32,shape=(len(full_train),8192))
    for start in range(0,len(full_train),512):
        stop = min(start+512,len(full_train))
        block = np.zeros((stop-start,8192),dtype=np.float32)
        for weight,arr in zip(checkpoint_weights,arrays):
            block += np.float32(weight) * arr[start:stop]
        assert np.isfinite(block).all()
        out[start:stop] = block
    out.flush()
    del out,arrays,block
    os.replace(tmp,aggregate_path)
    aggregation_metadata = {'identity':aggregation_identity,'sha256':sha256(aggregate_path),
                            'aggregation_seconds':time.perf_counter()-started}
    save_json(aggregate_meta_path,aggregation_metadata)
print('Checkpoint weights:', dict(zip(CHECKPOINT_STEPS,checkpoint_weights.tolist())))
print('Saved:',aggregate_path)


Checkpoint weights: {17: 0.5238095238095238, 34: 0.3537414965986394, 51: 0.12244897959183673}
Saved: /kaggle/working/HealthVer_TAGCOS_Experiments/aggregated_features.npy


In [12]:
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits
from scipy.linalg import lstsq

def allocate_quotas(cluster_counts, total):
    counts = np.asarray(cluster_counts,dtype=np.int64)
    assert (counts >= 0).all() and 0 < total <= counts.sum()
    numerators = counts * int(total)
    quotas = numerators // counts.sum()
    remainders = numerators % counts.sum()
    order = np.lexsort((np.arange(len(counts)), -remainders))
    for cluster_id in order[:int(total-quotas.sum())]:
        quotas[cluster_id] += 1
    assert quotas.sum() == total and (quotas <= counts).all()
    return quotas

def positive_omp_round(x, target, budget, lam=1.0, tol=1e-4):
    # x: examples x feature dimensions, NumPy float32 on CPU.
    assert x.ndim == 2 and target.shape == (x.shape[1],)
    assert 0 < budget <= len(x)
    target_norm = float(np.linalg.norm(target))
    if not np.isfinite(target_norm) or target_norm <= 1e-12:
        raise RuntimeError('Degenerate cluster centroid; inspect before changing the method.')
    residual = target.copy()
    active = []
    weights = np.empty(0,dtype=x.dtype)
    removed = 0
    for _ in range(budget):
        if float(np.linalg.norm(residual)) / target_norm < tol:
            break
        scores = x @ residual
        index = int(np.argmax(scores))
        # Preserve the reference's stop-on-reselected-atom behavior.
        if index in active or float(scores[index]) <= 0:
            break
        active.append(index)
        if len(active) == 1:
            denom = np.dot(x[index],x[index])
            assert float(denom) > 0
            weights = (scores[index] / denom).reshape(1)
        else:
            while active:
                atoms = x[active]
                gram = atoms @ atoms.T + np.float32(lam) * np.eye(len(active),dtype=x.dtype)
                weights = lstsq(gram,atoms @ target,lapack_driver='gelsy')[0]
                assert np.isfinite(weights).all()
                if float(weights.min()) >= 0:
                    break
                active.pop(int(np.argmin(weights)))
                removed += 1
            if not active:
                return [], removed
        residual = target - x[active].T @ weights
    # The upstream selector retains only nonzero coefficients.
    retained = [i for i,w in zip(active,weights.tolist()) if w > 0]
    return retained, removed

def select_cluster(x, target, quota):
    if quota == 0:
        return [], {'rounds':0,'removed_negative_coefficients':0}
    remaining = np.arange(len(x),dtype=np.int64)
    selected, rounds, removed = [], 0, 0
    while len(selected) < quota:
        need = quota - len(selected)
        support,dropped = positive_omp_round(x[remaining],target,need,
            lam=TAGCOS_CONFIG['omp']['lambda'],tol=TAGCOS_CONFIG['omp']['tol'])
        if not support:
            raise RuntimeError('OMP made no progress before reaching its quota; no fallback applied.')
        chosen = remaining[np.asarray(support,dtype=np.int64)]
        selected.extend(chosen.tolist())
        keep = np.ones(len(remaining),dtype=bool)
        keep[np.asarray(support,dtype=np.int64)] = False
        remaining = remaining[keep]
        rounds += 1
        removed += dropped
    assert len(selected) == len(set(selected)) == quota
    return selected, {'rounds':rounds,'removed_negative_coefficients':removed}

cluster_dir = TAGCOS_ROOT / 'clusters'
cluster_dir.mkdir(exist_ok=True)
cluster_file = cluster_dir / 'kmeans_clusters.npz'
cluster_meta_file = cluster_dir / 'cluster_metadata.json'
cluster_identity = {'protocol_sha256':PROTOCOL_HASH,
                    'aggregated_features_sha256':sha256(aggregate_path)}
X = np.load(aggregate_path,mmap_mode='r',allow_pickle=False)
assert X.shape == (len(full_train),8192) and np.isfinite(X).all()
if cluster_meta_file.exists():
    cluster_metadata = json.loads(cluster_meta_file.read_text())
    assert cluster_metadata['identity'] == cluster_identity
    assert cluster_metadata['sha256'] == sha256(cluster_file)
else:
    assert not cluster_file.exists(), 'Cluster file lacks completion metadata.'
    started = time.perf_counter()
    with threadpool_limits(limits=4):
        fitted = KMeans(**TAGCOS_CONFIG['kmeans']).fit(np.array(X,copy=True))
    np.savez(cluster_file,labels=fitted.labels_.astype(np.int64),centers=fitted.cluster_centers_.astype(np.float32))
    cluster_metadata = {'identity':cluster_identity,'sha256':sha256(cluster_file),
                        'clustering_seconds':time.perf_counter()-started,
                        'inertia':float(fitted.inertia_),'iterations':int(fitted.n_iter_)}
    save_json(cluster_meta_file,cluster_metadata)
    del fitted
with np.load(cluster_file,allow_pickle=False) as clustered:
    cluster_labels = clustered['labels'].copy()
    cluster_centers = clustered['centers'].copy()
assert cluster_labels.shape == (len(full_train),)
assert cluster_centers.shape == (100,8192) and np.isfinite(cluster_centers).all()
assert (cluster_labels >= 0).all() and (cluster_labels < 100).all()
counts = np.bincount(cluster_labels,minlength=100)
assert (counts > 0).all(), 'An empty cluster needs inspection.'
print('Cluster size range:',int(counts.min()),'to',int(counts.max()))

selections_dir = TAGCOS_ROOT / 'selections'
selections_dir.mkdir(exist_ok=True)
train_metadata = json.loads((BASELINE_ROOT/'dataset_metadata.json').read_text())
train_csv = BASELINE_ROOT/'original_data'/f"{train_metadata['revision']}_healthver_train.csv"
assert sha256(train_csv) == train_metadata['files']['train']['sha256']
train_frame = pd.read_csv(train_csv,dtype=str,keep_default_na=False)
assert train_frame['id'].tolist() == [str(i) for i in full_train['id']]
assert set(train_frame['label']).issubset(input_config['labels'])
summary = []
old_threads = torch.get_num_threads()
torch.set_num_threads(min(4,os.cpu_count() or 1))
try:
    for subset_name,size in SUBSET_SIZES.items():
        manifest_path = selections_dir / f'subset_{subset_name}.json'
        quotas = allocate_quotas(counts,size)
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text())
            assert manifest['protocol'] == TAGCOS_CONFIG
            assert manifest['aggregated_features_sha256'] == sha256(aggregate_path)
            assert manifest['clusters_sha256'] == sha256(cluster_file)
            assert manifest['cluster_quotas'] == quotas.tolist()
            print(subset_name, ': reusing saved selection')
        else:
            started = time.perf_counter()
            selected_rows, cluster_trace = [], []
            with threadpool_limits(limits=4):
                for c in tqdm(range(100),desc=f'OMP {subset_name}'):
                    row_ids = np.flatnonzero(cluster_labels == c)
                    if quotas[c]:
                        values = np.array(X[row_ids],dtype=np.float32,copy=True)
                        target = cluster_centers[c].copy()
                        local_indices,trace = select_cluster(values,target,int(quotas[c]))
                        selected_rows.extend(row_ids[local_indices].tolist())
                        del values,target
                    else:
                        trace = {'rounds':0,'removed_negative_coefficients':0}
                    cluster_trace.append({'cluster':c,'cluster_size':int(counts[c]),
                                          'quota':int(quotas[c]),**trace})
            indices = sorted(selected_rows)
            assert len(indices) == len(set(indices)) == size
            manifest = {'method':TAGCOS_CONFIG['method'],'subset':subset_name,
                'protocol':TAGCOS_CONFIG,'protocol_sha256':PROTOCOL_HASH,
                'dataset_fingerprint':full_train._fingerprint,
                'aggregated_features_sha256':sha256(aggregate_path),
                'clusters_sha256':sha256(cluster_file),
                'row_indices':indices,'ids':[str(i) for i in full_train.select(indices)['id']],
                'cluster_quotas':quotas.tolist(),'cluster_trace':cluster_trace,
                'selection_seconds':time.perf_counter()-started,
                'training_weights':'uniform'}
            save_json(manifest_path,manifest)
        indices = manifest['row_indices']
        assert len(indices) == len(set(indices)) == size and indices == sorted(indices)
        assert all(0 <= i < len(full_train) for i in indices)
        assert manifest['dataset_fingerprint'] == full_train._fingerprint
        assert manifest['ids'] == [str(i) for i in full_train.select(indices)['id']]
        assert np.array_equal(np.bincount(cluster_labels[indices],minlength=100),quotas)
        class_counts = train_frame.iloc[indices]['label'].value_counts()
        summary.append({'subset':subset_name,'rows':size,
                        **{label:int(class_counts.get(label,0)) for label in input_config['labels']}})
finally:
    torch.set_num_threads(old_threads)
del X
selection_summary = pd.DataFrame(summary)
selection_summary.to_csv(TAGCOS_ROOT/'selection_summary.csv',index=False)
print(selection_summary.to_string(index=False))
print('All selections fixed. Validation and test were not used for selection.')


Cluster size range: 13 to 349


OMP 50pct:   0%|          | 0/100 [00:00<?, ?it/s]

OMP 25pct:   0%|          | 0/100 [00:00<?, ?it/s]

OMP 10pct:   0%|          | 0/100 [00:00<?, ?it/s]

OMP 5pct:   0%|          | 0/100 [00:00<?, ?it/s]

subset  rows  Supports  Refutes  Neutral
 50pct  5295      1891     1208     2196
 25pct  2648       945      602     1101
 10pct  1059       378      242      439
  5pct   530       189      120      221
All selections fixed. Validation and test were not used for selection.


In [13]:
full_train = load_from_disk(str(BASELINE_ROOT/'tokenized_train'))
validation_tokens = load_from_disk(str(BASELINE_ROOT/'tokenized_validation'))
TRAIN_SUBSETS, SUBSET_MANIFESTS = {}, {}
for name,size in SUBSET_SIZES.items():
    manifest_path = TAGCOS_ROOT/'selections'/f'subset_{name}.json'
    manifest = json.loads(manifest_path.read_text())
    assert manifest['protocol'] == TAGCOS_CONFIG
    assert manifest['protocol_sha256'] == PROTOCOL_HASH
    assert manifest['dataset_fingerprint'] == full_train._fingerprint
    assert manifest['aggregated_features_sha256'] == sha256(aggregate_path)
    assert manifest['clusters_sha256'] == sha256(cluster_file)
    indices = manifest['row_indices']
    assert len(indices) == len(set(indices)) == size and indices == sorted(indices)
    assert all(0 <= i < len(full_train) for i in indices)
    selected = full_train.select(indices)
    assert [str(i) for i in selected['id']] == manifest['ids']
    TRAIN_SUBSETS[name] = selected.select_columns(['input_ids','attention_mask','labels'])
    SUBSET_MANIFESTS[name] = manifest
    print(f'{name}: {len(selected):,} training rows ready')
assert len(validation_tokens) == 1917
print('Shared validation:',len(validation_tokens))
print('Ready. No subset training or test evaluation performed.')


50pct: 5,295 training rows ready
25pct: 2,648 training rows ready
10pct: 1,059 training rows ready
5pct: 530 training rows ready
Shared validation: 1917
Ready. No subset training or test evaluation performed.


In [14]:
import gc
import json
import math
import time
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.nn.utils.rnn import pad_sequence
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, GenerationConfig,
    Trainer, TrainingArguments, set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import PeftModel
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)

# ---------- Fixed baseline settings ----------
DEVICE = torch.device("cuda:0")
assert torch.cuda.is_available()

input_config = json.loads(
    (BASELINE_ROOT / "input_config.json").read_text()
)
baseline_config = json.loads(
    (BASELINE_RUN_DIR / "run_config.json").read_text()
)
dataset_metadata = json.loads(
    (BASELINE_ROOT / "dataset_metadata.json").read_text()
)

LABELS = input_config["labels"]
SEED = baseline_config["seed"]
EPOCHS = baseline_config["epochs"]
MICRO_BATCH = baseline_config["micro_batch_size"]
GRAD_ACCUM = baseline_config["gradient_accumulation"]
INITIAL_ADAPTER = BASELINE_ROOT / "initial_adapter_seed42"

assert (SEED, EPOCHS, MICRO_BATCH, GRAD_ACCUM) == (42, 3, 8, 4)
assert baseline_config["learning_rate"] == 1e-4
assert INITIAL_ADAPTER.is_dir()

# Release GPU objects from scoring or earlier cells.
for object_name in [
    "warmup_trainer", "warmup_model", "trainer",
    "model", "base_model", "base",
]:
    globals().pop(object_name, None)
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    str(BASELINE_RUN_DIR / "best_adapter")
)
assert tokenizer.pad_token_id is not None

generation_config = GenerationConfig.from_pretrained(
    str(BASELINE_ROOT / "Qwen_ZeroShot_Validation")
)
assert generation_config.max_new_tokens == 12
assert not generation_config.do_sample
assert generation_config.num_beams == 1

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b""):
            h.update(chunk)
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)

def load_original(split, filename):
    path = BASELINE_ROOT / "original_data" / (
        f"{dataset_metadata['revision']}_{filename}"
    )
    assert sha256(path) == dataset_metadata["files"][split]["sha256"]
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert frame["id"].is_unique
    assert set(frame["label"]).issubset(LABELS)
    return frame

validation_frame = load_original("validation", "healthver_dev.csv")
test_frame = load_original("test", "healthver_test.csv")
assert len(validation_frame) == 1917
assert len(test_frame) == 1823
assert validation_frame["id"].tolist() == [
    str(uid) for uid in validation_tokens["id"]
]

# Verify every subset before starting any training.
aggregate_hash = sha256(TAGCOS_ROOT / "aggregated_features.npy")
manifests = {}

for name, size in SUBSET_SIZES.items():
    path = TAGCOS_ROOT / "selections" / f"subset_{name}.json"
    manifest = json.loads(path.read_text())
    assert manifest["protocol"] == TAGCOS_CONFIG
    assert manifest["aggregated_features_sha256"] == aggregate_hash
    assert manifest["protocol_sha256"] == PROTOCOL_HASH
    assert manifest["dataset_fingerprint"] == full_train._fingerprint
    indices = manifest["row_indices"]
    assert len(indices) == len(set(indices)) == size
    assert indices == sorted(indices)
    selected = full_train.select(indices)
    assert [str(uid) for uid in selected["id"]] == manifest["ids"]
    TRAIN_SUBSETS[name] = selected.select_columns(
        ["input_ids", "attention_mask", "labels"]
    )
    manifests[name] = manifest

initial_hashes = {
    path.name: sha256(path)
    for path in INITIAL_ADAPTER.iterdir()
    if path.name in (
        "adapter_config.json", "adapter_model.safetensors",
        "adapter_model.bin",
    )
}
assert "adapter_config.json" in initial_hashes
assert any("adapter_model" in name for name in initial_hashes)

# ---------- Same inputs and loss as baseline ----------
def make_messages(claim, evidence):
    return [
        {"role": "system", "content": input_config["system_prompt"]},
        {
            "role": "user",
            "content": f"Claim: {claim}\n\nEvidence: {evidence}",
        },
    ]

label_lookup = {label.lower(): label for label in LABELS}

def parse_prediction(text):
    normalized = " ".join(text.strip().lower().split())
    return label_lookup.get(normalized, "INVALID")

def training_collator(rows):
    padding = {
        "input_ids": tokenizer.pad_token_id,
        "attention_mask": 0,
        "labels": -100,
    }
    batch = {
        key: pad_sequence(
            [torch.tensor(row[key], dtype=torch.long) for row in rows],
            batch_first=True,
            padding_value=value,
        )
        for key, value in padding.items()
    }
    assert batch["input_ids"].shape[1] <= 384
    return batch

# ---------- Generated-answer validation and test ----------
def evaluate_labels(model, frame, folder, split, fingerprint):
    folder.mkdir(parents=True, exist_ok=True)
    previous_padding = tokenizer.padding_side
    previous_training = model.training
    records = []
    started = time.perf_counter()

    try:
        tokenizer.padding_side = "left"
        model.eval()

        with torch.inference_mode():
            for offset in range(0, len(frame), 8):
                rows = frame.iloc[offset:offset + 8]
                prompts = [
                    tokenizer.apply_chat_template(
                        make_messages(claim, evidence),
                        tokenize=False,
                        add_generation_prompt=True,
                    )
                    for claim, evidence in zip(
                        rows["claim"], rows["evidence"]
                    )
                ]
                inputs = tokenizer(
                    prompts,
                    padding=True,
                    truncation=False,
                    add_special_tokens=False,
                    return_tensors="pt",
                ).to(DEVICE)

                width = inputs["input_ids"].shape[1]
                limit = (
                    384 if split == "validation"
                    else model.config.max_position_embeddings
                )
                assert width + generation_config.max_new_tokens <= limit

                generated = model.generate(
                    **inputs, generation_config=generation_config
                )
                responses = tokenizer.batch_decode(
                    generated[:, width:], skip_special_tokens=True
                )

                for uid, gold, response in zip(
                    rows["id"], rows["label"], responses
                ):
                    records.append({
                        "id": uid,
                        "true_label": gold,
                        "predicted_label": parse_prediction(response),
                        "raw_response": response,
                    })
                del inputs, generated

        assert [row["id"] for row in records] == frame["id"].tolist()
        gold = [row["true_label"] for row in records]
        predicted = [row["predicted_label"] for row in records]

        metrics = {
            "evaluation_split": split,
            "n_examples": len(records),
            "run_config_sha256": fingerprint,
            "accuracy": float(accuracy_score(gold, predicted)),
            "macro_f1": float(f1_score(
                gold, predicted, labels=LABELS,
                average="macro", zero_division=0,
            )),
            "weighted_f1": float(f1_score(
                gold, predicted, labels=LABELS,
                average="weighted", zero_division=0,
            )),
            "invalid_rate": float(
                np.mean(np.asarray(predicted) == "INVALID")
            ),
            "elapsed_seconds": time.perf_counter() - started,
        }

        with (folder / "predictions.jsonl").open("w") as handle:
            for record in records:
                handle.write(json.dumps(record) + "\n")

        save_json(folder / "classification_report.json",
                  classification_report(
                      gold, predicted, labels=LABELS,
                      output_dict=True, zero_division=0,
                  ))

        matrix_labels = LABELS + ["INVALID"]
        pd.DataFrame(
            confusion_matrix(gold, predicted, labels=matrix_labels),
            index=matrix_labels,
            columns=matrix_labels,
        ).loc[LABELS].to_csv(folder / "confusion_matrix.csv")

        # Completion marker for evaluation is written last.
        save_json(folder / "metrics.json", metrics)
        return metrics

    finally:
        tokenizer.padding_side = previous_padding
        model.train(previous_training)
        gc.collect()
        torch.cuda.empty_cache()

class SingleGPUArguments(TrainingArguments):
    @property
    def n_gpu(self):
        return 1

class TAGCOSTrainer(Trainer):
    def evaluate(
        self, eval_dataset=None, ignore_keys=None,
        metric_key_prefix="eval", **kwargs
    ):
        result = evaluate_labels(
            self.model,
            validation_frame,
            Path(self.args.output_dir) / "validation"
            / f"step_{self.state.global_step}",
            "validation",
            self.run_fingerprint,
        )
        metrics = {
            f"{metric_key_prefix}_{key}": result[key]
            for key in [
                "accuracy", "macro_f1", "weighted_f1", "invalid_rate"
            ]
        }
        metrics[f"{metric_key_prefix}_runtime"] = result["elapsed_seconds"]
        self.log(metrics)
        self.control = self.callback_handler.on_evaluate(
            self.args, self.state, self.control, metrics
        )
        return metrics

# ---------- One independent run per subset ----------
def run_subset(name):
    run_dir = TAGCOS_ROOT / f"TAGCOS_{name}_seed42"
    run_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = TAGCOS_ROOT / "selections" / f"subset_{name}.json"
    train_data = TRAIN_SUBSETS[name]
    steps_per_epoch = math.ceil(len(train_data) / 32)

    run_config = {
        **baseline_config,
        "training_rows": len(train_data),
        "protocol_sha256": PROTOCOL_HASH,
        "method": "TAGCOS_HealthVer_adaptation",
        "subset": name,
        "selection_sha256": sha256(manifest_path),
        "initial_adapter_sha256": initial_hashes,
        "protocol": TAGCOS_CONFIG,
        "optimizer": "adamw_torch",
        "weight_decay": 0.01,
        "max_grad_norm": 1.0,
        "scheduler": "linear",
        "warmup_steps": math.ceil(0.03 * EPOCHS * steps_per_epoch),
        "generation_config": generation_config.to_dict(),
        "evaluation_batch_size": 8,
    }
    config_path = run_dir / "run_config.json"
    if config_path.exists():
        assert json.loads(config_path.read_text()) == run_config, (
            f"Existing configuration differs: {run_dir}"
        )
    else:
        save_json(config_path, run_config)

    fingerprint = hashlib.sha256(
        json.dumps(run_config, sort_keys=True).encode()
    ).hexdigest()
    completion_path = run_dir / "training_complete.json"
    test_path = run_dir / "test" / "metrics.json"

    complete = (
        json.loads(completion_path.read_text())
        if completion_path.exists() else None
    )
    if complete is not None:
        assert complete["run_config_sha256"] == fingerprint
        assert (run_dir / "best_adapter" / "adapter_config.json").exists()

    print(f"\nTAGCOS {name}: {len(train_data):,} rows")

    if complete is not None and test_path.exists():
        test_metrics = json.loads(test_path.read_text())
        assert test_metrics["run_config_sha256"] == fingerprint
        print("Completed run found; reusing saved results.")
        return complete, test_metrics

    local_base = local_model = local_trainer = None
    try:
        set_seed(SEED)
        tokenizer.padding_side = "right"

        local_base = AutoModelForCausalLM.from_pretrained(
            input_config["model"],
            revision=input_config["model_revision"],
            dtype=torch.float16,
            attn_implementation="sdpa",
        ).to(DEVICE)

        adapter_path = (
            run_dir / "best_adapter"
            if complete is not None else INITIAL_ADAPTER
        )
        local_model = PeftModel.from_pretrained(
            local_base,
            str(adapter_path),
            is_trainable=complete is None,
        )
        local_model.config.use_cache = False
        local_model.config.pad_token_id = tokenizer.pad_token_id
        local_model.gradient_checkpointing_disable()

        if complete is None:
            for parameter_name, parameter in local_model.named_parameters():
                if parameter.requires_grad:
                    assert "lora_" in parameter_name
                    parameter.data = parameter.data.float()
            del parameter
            assert sum(
                p.numel() for p in local_model.parameters()
                if p.requires_grad
            ) == 8798208

            args = SingleGPUArguments(
                output_dir=str(run_dir),
                num_train_epochs=EPOCHS,
                per_device_train_batch_size=MICRO_BATCH,
                gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=baseline_config["learning_rate"],
                warmup_steps=run_config["warmup_steps"],
                lr_scheduler_type="linear",
                optim="adamw_torch",
                weight_decay=0.01,
                max_grad_norm=1.0,
                fp16=True,
                bf16=False,
                gradient_checkpointing=False,
                eval_strategy="epoch",
                save_strategy="epoch",
                save_total_limit=2,
                save_only_model=False,
                load_best_model_at_end=True,
                metric_for_best_model="macro_f1",
                greater_is_better=True,
                logging_steps=25,
                logging_first_step=True,
                report_to="none",
                dataloader_num_workers=0,
                dataloader_pin_memory=True,
                seed=SEED,
                data_seed=SEED,
                label_names=["labels"],
            )
            assert args.world_size == 1
            assert args.device == DEVICE
            assert args.train_batch_size == 8
            (run_dir / "training_args.json").write_text(
                args.to_json_string()
            )

            local_trainer = TAGCOSTrainer(
                model=local_model,
                args=args,
                train_dataset=train_data,
                eval_dataset=validation_tokens,
                data_collator=training_collator,
                processing_class=tokenizer,
            )
            local_trainer.run_fingerprint = fingerprint

            checkpoint = get_last_checkpoint(str(run_dir))
            print("Starting from:", checkpoint or INITIAL_ADAPTER)
            print("Expected optimizer steps:", EPOCHS * steps_per_epoch)

            torch.cuda.reset_peak_memory_stats(DEVICE)
            result = local_trainer.train(
                resume_from_checkpoint=checkpoint
            )
            assert local_trainer.state.epoch >= EPOCHS - 1e-6
            assert local_trainer.state.global_step == EPOCHS * steps_per_epoch
            assert local_trainer.state.best_metric is not None

            # Trainer has restored the best validation checkpoint.
            local_trainer.save_model(str(run_dir / "best_adapter"))
            tokenizer.save_pretrained(run_dir / "best_adapter")
            local_trainer.save_metrics("train", result.metrics)
            local_trainer.save_state()

            complete = {
                "run_config_sha256": fingerprint,
                "best_validation_macro_f1": float(
                    local_trainer.state.best_metric
                ),
                "best_checkpoint": local_trainer.state.best_model_checkpoint,
                "epoch": float(local_trainer.state.epoch),
                "global_step": int(local_trainer.state.global_step),
                "train_call_runtime_seconds": result.metrics["train_runtime"],
                "peak_allocated_gib": (
                    torch.cuda.max_memory_allocated(DEVICE) / 1024**3
                ),
                "resumed_from": checkpoint,
                "runtime_note": (
                    "A resumed call excludes time spent in earlier sessions. "
                    "TAGCOS selection cost is recorded separately."
                ),
            }
            save_json(completion_path, complete)
            local_model = local_trainer.model

        test_metrics = evaluate_labels(
            local_model, test_frame, run_dir / "test",
            "test", fingerprint,
        )
        print(
            f"Best validation F1: "
            f"{complete['best_validation_macro_f1']:.4f} | "
            f"Test F1: {test_metrics['macro_f1']:.4f} | "
            f"Test accuracy: {test_metrics['accuracy']:.4f}"
        )
        return complete, test_metrics

    finally:
        local_trainer = local_model = local_base = None
        gc.collect()
        torch.cuda.empty_cache()

# ---------- Fixed loop: test results never change later runs ----------
summary_rows = []

for subset_name in ["50pct", "25pct", "10pct", "5pct"]:
    completion, metrics = run_subset(subset_name)
    summary_rows.append({
        "subset": subset_name,
        "training_rows": len(TRAIN_SUBSETS[subset_name]),
        "training_seed": SEED,
        "best_validation_macro_f1":
            completion["best_validation_macro_f1"],
        "test_accuracy": metrics["accuracy"],
        "test_macro_f1": metrics["macro_f1"],
        "test_weighted_f1": metrics["weighted_f1"],
        "test_invalid_rate": metrics["invalid_rate"],
        "train_call_seconds": completion["train_call_runtime_seconds"],
        "test_seconds": metrics["elapsed_seconds"],
    })

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(TAGCOS_ROOT / "tagcos_results.csv", index=False)
    print(summary.to_string(index=False))

print("\nAll four runs complete.")
print("Results:", TAGCOS_ROOT / "tagcos_results.csv")



TAGCOS 50pct: 5,295 rows


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting from: /kaggle/input/datasets/muusamunufo/base-1-trained/HealthVer_Experiments/initial_adapter_seed42
Expected optimizer steps: 498


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.193911,No log,0.704747,0.676981,0.711316,0.000000
2,0.119397,No log,0.779343,0.755442,0.785939,0.000000
3,0.051920,No log,0.795514,0.764325,0.797499,0.000000


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


... (12 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting from: /kaggle/input/datasets/muusamunufo/base-1-trained/HealthVer_Experiments/initial_adapter_seed42
Expected optimizer steps: 51


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Invalid Rate
1,0.339253,No log,0.558685,0.397728,0.496729,0.000000
2,0.348750,No log,0.603026,0.551154,0.602984,0.000000
3,0.246011,No log,0.627021,0.572777,0.623899,0.000000


Best validation F1: 0.5728 | Test F1: 0.5576 | Test accuracy: 0.5891
subset  training_rows  training_seed  best_validation_macro_f1  test_accuracy  test_macro_f1  test_weighted_f1  test_invalid_rate  train_call_seconds  test_seconds
 50pct           5295             42                  0.764325       0.746023       0.731860          0.746023                0.0           1067.1472     65.836873
 25pct           2648             42                  0.727722       0.722436       0.706441          0.724663                0.0            642.6532     65.745446
 10pct           1059             42                  0.631497       0.663193       0.640177          0.663017                0.0            384.5099     65.653037
  5pct            530             42                  0.572777       0.589139       0.557584          0.582725                0.0            298.8369     65.440896

All four runs complete.
Results: /kaggle/working/HealthVer_TAGCOS_Experiments/tagcos_results.csv


In [15]:
results = pd.read_csv(TAGCOS_ROOT/'tagcos_results.csv')
assert results['subset'].tolist() == list(SUBSET_SIZES)
assert results['training_rows'].tolist() == list(SUBSET_SIZES.values())
assert (results['training_seed'] == 42).all()
for name,size in SUBSET_SIZES.items():
    run = TAGCOS_ROOT/f'TAGCOS_{name}_seed42'
    config = json.loads((run/'run_config.json').read_text())
    completion = json.loads((run/'training_complete.json').read_text())
    metrics = json.loads((run/'test'/'metrics.json').read_text())
    fingerprint = json_hash(config)
    assert config['protocol_sha256'] == PROTOCOL_HASH
    assert config['selection_sha256'] == sha256(TAGCOS_ROOT/'selections'/f'subset_{name}.json')
    assert config['initial_adapter_sha256'] == initial_hashes
    assert completion['run_config_sha256'] == metrics['run_config_sha256'] == fingerprint
    assert completion['epoch'] >= 3 - 1e-6
    assert completion['global_step'] == 3*math.ceil(size/32)
    assert metrics['n_examples'] == 1823 and metrics['evaluation_split'] == 'test'
    assert (run/'best_adapter'/'adapter_config.json').is_file()
    assert (run/'best_adapter'/'adapter_model.safetensors').is_file()
    records = [json.loads(line) for line in (run/'test'/'predictions.jsonl').read_text().splitlines()]
    assert [r['id'] for r in records] == test_frame['id'].tolist()
    assert [r['true_label'] for r in records] == test_frame['label'].tolist()
    reconstructed = f1_score([r['true_label'] for r in records],
        [r['predicted_label'] for r in records],labels=LABELS,average='macro',zero_division=0)
    assert abs(reconstructed-metrics['macro_f1']) < 1e-12
    assert abs(float(results.loc[results['subset']==name,'test_macro_f1'].iloc[0])-reconstructed) < 1e-12

feature_seconds = sum(json.loads((TAGCOS_ROOT/'features'/f'checkpoint-{s}'/'feature_metadata.json').read_text())['extraction_seconds']
                      for s in CHECKPOINT_STEPS)
aggregation_seconds = json.loads(aggregate_meta_path.read_text())['aggregation_seconds']
clustering_seconds = json.loads(cluster_meta_file.read_text())['clustering_seconds']
omp_seconds = {name:json.loads((TAGCOS_ROOT/'selections'/f'subset_{name}.json').read_text())['selection_seconds']
               for name in SUBSET_SIZES}
warmup_runtime, warmup_runtime_source = None, None
for candidate in (WARMUP_DIR/'train_results.json',WARMUP_DIR/'warmup_metrics.json',WARMUP_DIR/'training_metrics.json'):
    if candidate.exists():
        value = json.loads(candidate.read_text())
        if 'train_runtime' in value:
            warmup_runtime = float(value['train_runtime'])
            warmup_runtime_source = str(candidate)
            break
if warmup_runtime is None:
    trainer_state_path = WARMUP_DIR/'trainer_state.json'
    if trainer_state_path.exists():
        for entry in json.loads(trainer_state_path.read_text()).get('log_history',[]):
            if 'train_runtime' in entry:
                warmup_runtime = float(entry['train_runtime'])
                warmup_runtime_source = str(trainer_state_path)
shared_selection = feature_seconds+aggregation_seconds+clustering_seconds
cost_report = {
    'protocol_sha256':PROTOCOL_HASH,'warmup_seconds':warmup_runtime,
    'warmup_runtime_source':warmup_runtime_source,
    'feature_extraction_seconds':feature_seconds,
    'aggregation_seconds':aggregation_seconds,'clustering_seconds':clustering_seconds,
    'omp_seconds_by_subset':omp_seconds,
    'selection_seconds_by_subset_including_shared_work':{
        name:shared_selection+seconds+(warmup_runtime or 0) for name,seconds in omp_seconds.items()},
    'all_budgets_selection_seconds_with_shared_work_counted_once':
        shared_selection+sum(omp_seconds.values())+(warmup_runtime or 0),
    'warmup_included':warmup_runtime is not None,
    'timing_note':'Persisted runtimes; excludes feature model loading and uncommitted interrupted work. '
                  'Subset train_call_seconds includes validation; resumed calls exclude previous sessions. '
                  'Missing warmup time is explicitly excluded, not assumed zero.',
}
save_json(TAGCOS_ROOT/'selection_costs.json',cost_report)
save_json(TAGCOS_ROOT/'final_audit.json',{
    'status':'PASSED','protocol_sha256':PROTOCOL_HASH,
    'results_sha256':sha256(TAGCOS_ROOT/'tagcos_results.csv'),
    'checked_subsets':list(SUBSET_SIZES),'test_rows':1823,'training_seed':42,
    'claim_scope':'Artifact consistency for one final-training seed; not statistical significance.',
})
print(results.to_string(index=False))
print('\nSelection costs:',json.dumps(cost_report,indent=2))
print('\nFINAL ARTIFACT AUDIT PASSED. Ready to save/commit this version with outputs.')
print('Results:',TAGCOS_ROOT/'tagcos_results.csv')


subset  training_rows  training_seed  best_validation_macro_f1  test_accuracy  test_macro_f1  test_weighted_f1  test_invalid_rate  train_call_seconds  test_seconds
 50pct           5295             42                  0.764325       0.746023       0.731860          0.746023                0.0           1067.1472     65.836873
 25pct           2648             42                  0.727722       0.722436       0.706441          0.724663                0.0            642.6532     65.745446
 10pct           1059             42                  0.631497       0.663193       0.640177          0.663017                0.0            384.5099     65.653037
  5pct            530             42                  0.572777       0.589139       0.557584          0.582725                0.0            298.8369     65.440896

Selection costs: {
  "protocol_sha256": "50ed943149137753976ffd395d6b91a07c162fe4f03499faa6e36744a3de1180",
  "warmup_seconds": 135.4039,
  "warmup_runtime_source": "/kaggle/worki